V'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Coca-Cola_FEMSA_Green_Bond_Framework.pdf'

In [8]:
import pymupdf
import spacy
import re
import pandas as pd
import unicodedata
import os
from pathlib import Path

In [9]:
nlp = spacy.load('en_core_web_lg')

In [10]:
# word vectors (similarity) vs linguistic featues e.g. lemma e.g. transportation vs transport

renewable_energy = ['renewable', 'solar', 'wind', 'bioenergy', 'biofuel', 'biomass', 'hydropower', 'hydrogen', 'power', 'grid', 'transmission', 'generation']
energy_efficiency = ['efficiency', 'retrofit']
pollution_prevention_and_control = ['pollution', 'waste']
environmentally_sustainable_management_of_living_natural_resources_and_land_use = ['land', 'agriculture', 'forestry', 'forest', 'fisheries', 'food']
terrestrial_and_aquatic_biodiversity_conservation = ['terrestrial', 'aquatic', 'biodiversity', 'conservation']
clean_transportation = ['transportation', 'electric', 'battery', 'EV', 'charger', 'bus', 'rail', 'train', 'car', 'vehicle', 'bicycle', 'non-motorized', 'aviation']
sustainable_water_and_wastewater_management = ['water', 'potable', 'wastewater', 'sanitation', 'treatment']
climate_change_adaptation = ['adaptation', 'disaster']
circular_economy_and_or_ecoefficient_projects = ['circular', 'recycle', 'reuse']
green_buildings = ['buildings', 'appliances']

In [11]:
# prod code

def find_uop(document, language):

    pdf = pymupdf.open(document)

    if language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria']

    elif language == 'PT':
        keywordsUOP = ['Uso de Recursos']
        keywordsSEEGP = ['Processo de Avaliação e Seleção de Projetos']

    elif language == 'ES':
        keywordsUOP = ['Uso de fondos', 'Uso de los fondos']
        keywordsSEEGP = ['XYZ']

    areaUOP = None
    areaSEEGP = None

    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

        if areaUOP is None:
            for keyword in keywordsUOP:
                start = page.search_for(keyword)
                if start:
                    areaUOP = (page_idx, start[0])

        if areaSEEGP is None:
            for keyword in keywordsSEEGP:
                end = page.search_for(keyword)
                if end:
                    areaSEEGP = (page_idx, end[0])

    return areaUOP, areaSEEGP

In [12]:
# iterate pages in the pdf and extract preferrably tables or else words from UOP section

def page_scenario_and_extract(document, areaUOP, areaSEEGP):

    pdf = pymupdf.open(document)
    # language = language

    # inputs
    start_page_idx = areaUOP[0]
    end_page_idx = areaSEEGP[0]
    start_point = areaUOP[1].y1
    end_point = areaSEEGP[1].y0

    # outputs
    noTableMsg = []
    hasTableMsg = []
    hasDFMsg = []
    extractUOPwords = []

# iterate pages
    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

# process four page scenarios, check for tables, extract tables else extract text as words
    # A: UOP all on a single page
        if start_page_idx == end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point and bbox[3] < end_point:
                    tableAheader = tables[0].header.names
                    tableAdf = tables[0].to_pandas()
                hasTableMsg.append(tableAheader)
                hasDFMsg.append(tableAdf)
            else:
                noTableMsg.append('No tableA')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point and y1 < end_point:
                        extractUOPwords.append(text)

    # UOP across >1 page
    # B: current page is start page
        elif page_idx == start_page_idx and page_idx == end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point:
                    tableBheader = tables[0].header.names
                    tableBdf = tables[0].to_pandas()
                hasTableMsg.append(tableBheader)
                hasDFMsg.append(tableBdf)
            else:
                noTableMsg.append('No tableB')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point:
                        extractUOPwords.append(text)

    # D: current page is neither start nor end page but is in the UOP area
        elif page_idx > start_page_idx and page_idx < end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                tableDheader = tables[0].header.names
                tableDdf = tables[0].to_pandas()
                hasTableMsg.append(tableDheader)
                hasDFMsg.append(tableDdf)
            else:
                noTableMsg.append(page_idx)
                noTableMsg.append('No tableD')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    extractUOPwords.append(text)

    # C: current page is end page
        elif page_idx == end_page_idx:
            tables = page.find_tables()
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[3] < end_point:
                    tableCheader = tables[0].header.names
                    tableCdf = tables[0].to_pandas()
                hasTableMsg.append(tableCheader)
                hasDFMsg.append(tableCdf)
            else:
                noTableMsg.append('No tableC')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y1 < end_point:
                        extractUOPwords.append(text)

    return noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords

    # noTableMsg lists page scenarios (and idx for scenario D) where no table is found
    # hasTableMsg lists the header row for found tables
    # hasDFMsg dataframe content in a list

In [13]:
# check extracted table is UOP table and extract Categories

def UOP_table_cats(hasDFMsg, hasTableMsg):

    # inputs
    hasDFMsg = hasDFMsg
    hasTableMsg = hasTableMsg

    # outputs
    uniqueCats = []
    tableInfo = []

    if len(hasDFMsg) >= 1:
        tableInfo.append('More than one table found')
    

    # this over simplifies by assuming the category is always in the first column
    # the header condition code oversimplifies by assuming table fragments split across pages without a header row contain no new category labels

        if 'Category' in hasTableMsg[0]:
            x = 'singular'
        elif 'Categories' in hasTableMsg[0]:
            x = 'plural'
        elif 'Eligible Project Category' in hasTableMsg[0]:
            x = 'phrase'
        else:
            x = 'not a UOP table or category not in 1st columns'

        DFname = hasDFMsg[0]
        if x == 'singular':
            uniqueC = DFname['Category'].unique()
            uniqueCats.append(uniqueC)
        elif x == 'plural':
            uniqueC = DFname['Categories'].unique()
            uniqueCats.append(uniqueC)
        elif x == 'phrase':
            uniqueC = DFname['Eligible Project Category'].unique()
            uniqueCats.append(uniqueC)
        else:
            tableInfo.append(x)

    if len(hasDFMsg) == 0:
        tableInfo.append('No table found')

    return tableInfo, uniqueCats

    # turn these into assert and proper error msgs later
        # if errorMsg is empty and uniqueCats contains a list of category like words, pdf has processed successfully
        # if errorMsg is not empty, there may be more than one table found, in which case DFname variable could be inaccurate
        # if errorMsg is not empty, the words Category or Categories may not be present in the header row, in which case may not be a UOP table or may be other words such as criteria


In [14]:
# process words into category word dataframes where token.similarity score passes threshold

def UOPwords_to_Catwords(extractUOPwords):

    # inputs
    extractUOPwords = extractUOPwords

    re = renewable_energy
    ee = energy_efficiency
    ppc = pollution_prevention_and_control
    esml = environmentally_sustainable_management_of_living_natural_resources_and_land_use
    tabc = terrestrial_and_aquatic_biodiversity_conservation
    ct = clean_transportation
    swwm = sustainable_water_and_wastewater_management
    cca = climate_change_adaptation
    ce = circular_economy_and_or_ecoefficient_projects
    gb = green_buildings

    similarity_threshold = 0.72

    # outputs
    simsre = {}
    simsee = {}
    simsppc = {}
    simsesml = {}
    simstabc = {}
    simsct = {}
    simsswwm = {}
    simscca = {}
    simsce = {}
    simsgb = {}


    # renewable_energy
    for worda in re:
        doca = nlp(worda)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doca.similarity(docb) >= similarity_threshold:
                sim = doca.similarity(docb)
                simsre[doca[0].text + ' ' + docb[0].text] = sim
                dfre = pd.DataFrame.from_dict(simsre, 'index')
            else:
                dfre = 'is not re'


    # energy_efficiency
    for wordc in ee:
        docc = nlp(wordc)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docc.similarity(docb) >= similarity_threshold:
                sim = docc.similarity(docb)
                simsee[docc[0].text + ' ' + docb[0].text] = sim
                dfee = pd.DataFrame.from_dict(simsee, 'index')
            else:
                dfee = 'is not ee'
    
    # pollution_prevention_and_control
    for wordd in ppc:
        docd = nlp(wordd)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docd.similarity(docb) >= similarity_threshold:
                sim = docd.similarity(docb)
                simsppc[docd[0].text + ' ' + docb[0].text] = sim
                dfppc = pd.DataFrame.from_dict(simsppc, 'index')
            else:
                dfppc = 'is not ppc'

    # environmentally_sustainable_management_of_living_natural_resources_and_land_use
    for worde in esml:
        doce = nlp(worde)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doce.similarity(docb) >= similarity_threshold:
                sim = doce.similarity(docb)
                simsesml[doce[0].text + ' ' + docb[0].text] = sim
                dfesml = pd.DataFrame.from_dict(simsesml, 'index')
            else:
                dfesml = 'is not esml'

    # terrestrial_and_aquatic_biodiversity_conservation
    for wordf in tabc:
        docf = nlp(wordf)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docf.similarity(docb) >= similarity_threshold:
                sim = docf.similarity(docb)
                simstabc[docf[0].text + ' ' + docb[0].text] = sim
                dftabc = pd.DataFrame.from_dict(simstabc, 'index')
            else:
                dftabc = 'is not tabc'

    # clean_transportation
    for wordg in ct:
        docg = nlp(wordg)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docg.similarity(docb) >= similarity_threshold:
                sim = docg.similarity(docb)
                simsct[docg[0].text + ' ' + docb[0].text] = sim
                dfct = pd.DataFrame.from_dict(simsct, 'index')
            else:
                dfct = 'is not ct'

    # sustainable_water_and_wastewater_management
    for wordh in swwm:
        doch = nlp(wordh)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doch.similarity(docb) >= similarity_threshold:
                sim = doch.similarity(docb)
                simsswwm[doch[0].text + ' ' + docb[0].text] = sim
                dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')
            else:
                dfswwm = 'is not swwm'

    # climate_change_adaptation
    for wordi in cca:
        doci = nlp(wordi)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doci.similarity(docb) >= similarity_threshold:
                sim = doci.similarity(docb)
                simscca[doci[0].text + ' ' + docb[0].text] = sim
                dfcca = pd.DataFrame.from_dict(simscca, 'index')
            else:
                dfcca = 'is not cca'

    # circular_economy_and_or_ecoefficient_projects
    for wordj in ce:
        docj = nlp(wordj)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docj.similarity(docb) >= similarity_threshold:
                sim = docj.similarity(docb)
                simsce[docj[0].text + ' ' + docb[0].text] = sim
                dfce = pd.DataFrame.from_dict(simsce, 'index')
            else:
                dfce = 'is not ce'

    # green_buildings
    for wordk in gb:
        dock = nlp(wordk)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if dock.similarity(docb) >= similarity_threshold:
                sim = dock.similarity(docb)
                simsgb[dock[0].text + ' ' + docb[0].text] = sim
                dfgb = pd.DataFrame.from_dict(simsgb, 'index')
            else:
                dfgb = 'is not gb'

    return simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb

In [16]:
# run the program
# notable

document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/refinement_reports/FS_Green_Bond_Framework_-_July_2021.pdf'
language = 'EN'
areaUOP, areaSEEGP = find_uop(document, language)
noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords = page_scenario_and_extract(document, areaUOP, areaSEEGP)
tableInfo, uniqueCats = UOP_table_cats(hasDFMsg, hasTableMsg)
if len(uniqueCats) == 0:
    simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb = UOPwords_to_Catwords(extractUOPwords)
filepath = Path(document)

/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_98869/2765828476.py:39: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doca.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_98869/2765828476.py:52: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docc.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_98869/2765828476.py:64: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docd.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_98869/2765828476.py:76: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doce.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_98869/2765828476.py:88: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docf.similarity(docb) 

In [17]:
print(hasTableMsg)
print(hasDFMsg)
print(hasDFMsg)

# print(hasTableMsg[0])
# print(len(extractUOPwords))

[]
[]
[]


In [18]:
print(areaUOP)
print(areaSEEGP)


(11, Rect(110.63999938964844, 441.1795349121094, 218.39576721191406, 455.7480773925781))
(12, Rect(110.63999938964844, 423.4195556640625, 406.753173828125, 437.98809814453125))


In [19]:
# collate the outputs
# notable

print(filepath.name, type(filepath.name))
print(language, type(language))
print(areaUOP[0], type(areaUOP[0]))
print(areaSEEGP[0], type(areaSEEGP[0]))
if len(hasDFMsg) != 0:
    if len(tableInfo) == 1:   
        print(tableInfo[0], type(tableInfo[0]))
    if len(tableInfo) == 2:
        print(tableInfo[1], type(tableInfo[1]))
    if len(uniqueCats) > 0:
        print(uniqueCats, len(uniqueCats))
    else:
        print('no categories from tables')
if len(hasDFMsg) == 0:
    print(f"renewable_energy  {simsre}")
    print(f"energy_efficiency {simsee}")
    print(f"pollution_prevention_and_control {simsppc}")
    print(f"environmentally_sustainable_management_of_living_natural_resources_and_land_use {simsesml}")
    print(f"terrestrial_and_aquatic_biodiversity_conservation {simstabc}")
    print(f"clean_transportation {simsct}")
    print(f"sustainable_water_and_wastewater_management {simsswwm}")
    print(f"climate_change_adaptation {simscca}")
    print(f"circular_economy_and_or_ecoefficient_projects {simsce}")
    print(f"green_buildings {simsgb}")
else:
    print('categories from tables')

FS_Green_Bond_Framework_-_July_2021.pdf <class 'str'>
EN <class 'str'>
11 <class 'int'>
12 <class 'int'>
renewable_energy  {'bioenergy biofuel': 0.7225276827812195, 'biofuel biofuel': 1.0, 'biomass biomass': 1.0, 'generation generation': 1.0}
energy_efficiency {}
pollution_prevention_and_control {}
environmentally_sustainable_management_of_living_natural_resources_and_land_use {'forestry Forestry': 0.784554123878479, 'forest forests': 0.8802252411842346, 'forest forest': 1.0, 'forest (': 0.7941798567771912}
terrestrial_and_aquatic_biodiversity_conservation {'conservation conservation': 0.7817807793617249}
clean_transportation {}
sustainable_water_and_wastewater_management {}
climate_change_adaptation {}
circular_economy_and_or_ecoefficient_projects {}
green_buildings {}


In [75]:
print(extractUOPwords)

['12', 'The', 'proceeds', 'to', 'be', 'raised', 'through', 'green', 'bond', 'issuances', 'will', 'be', 'used', 'to', 'finance', 'projects', 'in', 'the', 'following', 'categories:', 'Bioenergy1,2:', 'Projects', 'related', 'to', 'production', 'of', 'hydrous', 'and', 'anhydrous', 'corn-ethanol', 'biofuel,', 'including:', 'i.', 'capital', 'expenditures', 'for', 'development,', 'construction,', 'operation', 'and', 'maintenance', 'of', 'biofuel', 'production', 'facilities;', 'or', 'ii.', 'operational', 'expenditures', 'or', 'refinance', 'of', 'purchased', 'corn', 'feedstock', 'for', 'biofuel', 'production.', 'Feedstock', 'will', 'be', 'purchased', 'from', 'suppliers', 'in', 'compliance', 'with', 'FS', 'Sustainability', 'Protocol', 'and/or', 'certified', 'against', 'the', 'Climate', 'Bonds', 'Standard', 'Agriculture', 'Criteria', 'Version', '1.', '(1)', 'Excludes', 'fossil', 'biofuel', 'production', 'and', 'blending', 'facilities.', '(2)', 'The', 'hydrated', 'ethanol', 'is', 'the', 'common', 

In [123]:
import string

extractUOPwords_cleaned = []
remove_punct = str.maketrans('','',string.punctuation)
for word in extractUOPwords:
    cleaned_word = word.translate(remove_punct).strip()
    if cleaned_word:
        extractUOPwords_cleaned.append(cleaned_word)
print(extractUOPwords_cleaned)
print(type(extractUOPwords_cleaned))


['12', 'The', 'proceeds', 'to', 'be', 'raised', 'through', 'green', 'bond', 'issuances', 'will', 'be', 'used', 'to', 'finance', 'projects', 'in', 'the', 'following', 'categories', 'Bioenergy12', 'Projects', 'related', 'to', 'production', 'of', 'hydrous', 'and', 'anhydrous', 'cornethanol', 'biofuel', 'including', 'i', 'capital', 'expenditures', 'for', 'development', 'construction', 'operation', 'and', 'maintenance', 'of', 'biofuel', 'production', 'facilities', 'or', 'ii', 'operational', 'expenditures', 'or', 'refinance', 'of', 'purchased', 'corn', 'feedstock', 'for', 'biofuel', 'production', 'Feedstock', 'will', 'be', 'purchased', 'from', 'suppliers', 'in', 'compliance', 'with', 'FS', 'Sustainability', 'Protocol', 'andor', 'certified', 'against', 'the', 'Climate', 'Bonds', 'Standard', 'Agriculture', 'Criteria', 'Version', '1', '1', 'Excludes', 'fossil', 'biofuel', 'production', 'and', 'blending', 'facilities', '2', 'The', 'hydrated', 'ethanol', 'is', 'the', 'common', 'ethanol', 'sold', 

In [ ]:
# sandbox
# inputs
extractUOPwords = extractUOPwords

re = renewable_energy
retest = ['biofuel', 'forestry', 'forest']
ee = energy_efficiency
ppc = pollution_prevention_and_control
esml = environmentally_sustainable_management_of_living_natural_resources_and_land_use
tabc = terrestrial_and_aquatic_biodiversity_conservation
ct = clean_transportation
swwm = sustainable_water_and_wastewater_management
cca = climate_change_adaptation
ce = circular_economy_and_or_ecoefficient_projects
gb = green_buildings

similarity_threshold = 0.72

# outputs
simsre = {}
simsee = {}
simsppc = {}
simsesml = {}
simstabc = {}
simsct = {}
simsswwm = {}
simscca = {}
simsce = {}
simsgb = {}

extractUOPTest = ['biofuel', 'dog', 'cat', 'lion', 'forestry', 'forest']

# renewable_energy
for worda in retest:
    doca = nlp(worda)
    for wordb in extractUOPTest:
        docb = nlp(wordb)
        if doca.similarity(docb) >= similarity_threshold:
            sim = doca.similarity(docb)
            simsre[doca[0].text + ' ' + docb[0].text] = sim
            dfre = pd.DataFrame.from_dict(simsre, 'index')
        else:
            dfre = 'is not re'

 # environmentally_sustainable_management_of_living_natural_resources_and_land_use
for worde in retest:
    doce = nlp(worde)
    for wordb in extractUOPTest:
        docb = nlp(wordb)
        if doce.similarity(docb) >= similarity_threshold:
            sim = doce.similarity(docb)
            simsesml[doce[0].text + ' ' + docb[0].text] = sim
            dfesml = pd.DataFrame.from_dict(simsesml, 'index')
        else:
            dfesml = 'is not esml'

In [122]:
print(dfre)
print(dfesml)

                     0
biofuel biofuel    1.0
forestry forestry  1.0
forest forest      1.0
                     0
biofuel biofuel    1.0
forestry forestry  1.0
forest forest      1.0


In [86]:
x = 'biofuel'
y = 'biofuel'

xx = nlp(x)
yy = nlp(y)

xx.similarity(yy)


1.0

In [89]:
try:
    index = extractUOPwords.index('biofuel')
    print(index)

except ValueError:
    print('not found')

42
